# 🎤 Flow AI — Vocais reais GRÁTIS (ACE-Step no Colab)

Gera **música completa cantada** de graça usando GPU gratuita do Colab + modelo open-source **ACE-Step 1.5** (qualidade próxima ao Suno).

**Passo a passo:**
1. No Colab: *Tempo de execução → Alterar tipo de ambiente → GPU (T4 grátis)*
2. Rode as células **1 → 4** em ordem (a instalação demora ~10–20 min só na 1ª vez)
3. Gere sua música na célula **4** e baixe o MP3
4. (Opcional) Célula **5**: expõe o ACE-Step via URL pública → cole no `.env` do Flow (`ACESTEP_API_URL=`) e o app passa a gerar com vocais reais direto no botão Criar

> Sem Colab? Teste grátis sem instalar nada: https://acemusic.ai (oficial, 100% grátis) ou a demo https://huggingface.co/spaces/ACE-Step/Ace-Step-v1.5
>
> Projeto original: https://github.com/ace-step/ACE-Step-1.5 (licença MIT)

## 1️⃣ Verificar GPU grátis

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2️⃣ Instalar ACE-Step 1.5 (só na primeira vez — demora)

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!git clone https://github.com/ace-step/ACE-Step-1.5.git
!pip install -q requests pyngrok
%cd ACE-Step-1.5
!uv sync

## 3️⃣ Subir o servidor de música (roda em 2º plano)

In [ ]:
import time, requests

# Inicia o servidor da API (porta 8001) em background
!nohup uv run acestep-api > api.log 2>&1 &

# Aguarda ficar pronto (baixa os modelos na 1ª vez — pode demorar)
for i in range(60):
    try:
        r = requests.get('http://localhost:8001/health', timeout=5)
        if r.ok:
            print('✅ ACE-Step pronto!', r.json())
            break
    except Exception:
        pass
    time.sleep(10)
    if i % 6 == 5:
        print(f'... aguardando ({(i+1)*10}s)')
else:
    print('❌ Não subiu. Veja o log:')
    !tail -50 api.log

## 4️⃣ Gerar música com vocais (edite e rode)

In [ ]:
# @title 🎵 Sua música
PROMPT = 'romantic pop ballad with piano and strings'  # @param {type:"string"}
LYRICS = '''[Verse 1]\nAcordo cedo com o sol na janela\nO café na mesa e a vida singela\n[Chorus]\nCanta comigo, deixa o som te levar\nEssa é a nossa hora de brilhar'''  # @param {type:"string"}
DURATION = 60  # @param {type:"slider", min:15, max:180, step:5}
BPM = 90  # @param {type:"integer"}

import json, time, requests
from IPython.display import Audio, display
from google.colab import files

API = 'http://localhost:8001'

# 1. Envia a tarefa (2 variações de uma vez)
task = requests.post(f'{API}/release_task', json={
    'prompt': PROMPT,
    'lyrics': LYRICS,
    'vocal_language': 'pt',
    'audio_format': 'mp3',
    'audio_duration': DURATION,
    'bpm': BPM,
    'batch_size': 2,
}).json()
task_id = task['data']['task_id']
print('🎼 Gerando...', task_id)

# 2. Aguarda concluir
result = None
for _ in range(100):
    time.sleep(6)
    q = requests.post(f'{API}/query_result', json={'task_id_list': [task_id]}).json()
    job = q['data'][0]
    if job['status'] == 2:
        raise RuntimeError('Falhou: ' + str(job))
    if job['status'] == 1:
        result = json.loads(job['result'])
        break
    print('.', end='', flush=True)
print('\n✅ Pronto!')

# 3. Baixa as variações e toca aqui mesmo
for i, item in enumerate(result[:2]):
    dl = requests.get(f"{API}{item['file']}")
    fname = f'flow_vocal_v{i+1}.mp3'
    open(fname, 'wb').write(dl.content)
    print(f'🎧 Variação {i+1}:')
    display(Audio(fname))
    files.download(fname)

## 5️⃣ (Opcional) Conectar ao app Flow — gere com vocais no botão Criar

In [ ]:
# @title 🔗 Expõe o ACE-Step pra internet (precisa de token grátis do ngrok)
# Crie conta grátis em https://dashboard.ngrok.com e cole o authtoken:
NGROK_TOKEN = ''  # @param {type:"string"}

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(8001, 'http')
print('📋 Cole esta URL no .env do Flow AI:')
print(f'ACESTEP_API_URL={tunnel.public_url}')
print('\nDepois rode o Flow com npm run dev — o botão Criar passa a usar vocais reais.')